# Phase 03 — Descriptor-based Gradient Boosting

This notebook evaluates **one frozen cohort at a time** using RDKit descriptors, Morgan fingerprints, MACCS keys and optional 2D Mordred descriptors. It always reads the original MongoDB documents through the shared cleaning loader. Neither the manifest files nor the historical notebooks are changed.

The five stored outer folds provide the out-of-fold (OOF) evaluation. Their saved inner validation IDs are read without resplitting; each inner training set is reconstructed as the complement of its validation IDs. Column filtering, missing-value imputation, mutual information and permutation importance are all fitted **inside** each inner training fold.

For `Tg_md_common`, the notebook automatically evaluates **structural only**, **structural + bulk**, **structural + single-chain**, and **structural + bulk delta** on the same 135 samples and frozen folds. Optional MD-only ablations evaluate bulk, single-chain and bulk-delta separately. Bulk and single-chain descriptors are never combined. Other cohorts run structural only. MD helpers are appended to `gbr.py`; the original structural functions remain unchanged.


## 1. Configuration

Choose a cohort and whether to add Mordred descriptors. Set `RUN_FULL_BENCHMARK = True` only after checking the manifest and runtime budget. The original hyperparameter grid has 54 candidates per inner fold, so the full 5 × 5 nested CV is computationally demanding, especially for PolyInfo. First run the disabled-training validation cells; then test execution separately with a reduced grid, never use outer-test performance to choose hyperparameters.

`Tg_md_common` automatically runs four paired arms: structural, structural + bulk, structural + single-chain, and structural + bulk delta. The last arm uses the three bulk predictors at 300 K plus the 350 K minus 250 K change of each one (six MD columns, with no absolute 250 K/350 K predictors). Set `RUN_MD_ABLATIONS = True` to additionally evaluate MD bulk-only, single-chain-only, and bulk-delta-only. Other cohorts run structural only. Each arm receives a separate run log, and the structural arm still uses the original `benchmark_gbr()` unchanged. Previously completed runs remain valid, although the notebook retrains them unless you explicitly restrict `REPRESENTATIONS`.


In [4]:
import hashlib
import sys
import tempfile
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display
from pymongo import MongoClient
from dotenv import load_dotenv

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'runlog.py').exists() and (p / 'src/final_comparison/datasets.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Run the notebook from the polymer repository')
sys.path.insert(0, str(ROOT))

from src.final_comparison.datasets import COLLECTIONS, DB_NAME, MD_COLUMNS, load_cohort
from src.final_comparison.splits import load_manifest, fold_ids
from src.final_comparison.gbr import (
    PARAM_GRID, benchmark_gbr, validate_frozen_splits,
    MD_MODES, benchmark_gbr_md, load_md_features, md_source_hashes, build_bulk_delta_features,
)
from runlog import RunLogger

load_dotenv(ROOT / '.env')

COHORT = 'Tg_md_common'  # Tg_md_common, Tg_bicerano, Tg_polyinfo, or one of four other properties
USE_MORDRED = False
RUN_MD_ABLATIONS = False  # Additional MD-only bulk, single-chain, and bulk-delta ablations
RUN_FULL_BENCHMARK = True
SEED = 28

COHORTS = {
    'Tg_md_common': ('bicerano', 'Tg', True),
    'Tg_bicerano': ('bicerano', 'Tg', False),
    'Tg_polyinfo': ('polyinfo', 'Tg', False),
    'density_extended': ('extended', 'density', False),
    'Tc_extended': ('extended', 'Tc', False),
    'glass_cte_bicerano': ('bicerano', 'glass_cte', False),
    'rubber_cte_bicerano': ('bicerano', 'rubber_cte', False),
}
dataset, target, require_md_coverage = COHORTS[COHORT]
REPRESENTATIONS = ['structural']
if require_md_coverage:
    REPRESENTATIONS.extend([
        'structural_bulk', 'structural_single_chain', 'structural_bulk_delta',
    ])
    if RUN_MD_ABLATIONS:
        REPRESENTATIONS.extend(['md_bulk', 'md_single_chain', 'md_bulk_delta'])

# temp
REPRESENTATIONS = ['md_bulk', 'md_single_chain', 'md_bulk_delta']

assert all(r == 'structural' or r in MD_MODES for r in REPRESENTATIONS)

MD_DIR = ROOT / 'data/md_features'
MANIFEST = ROOT / 'data/splits/v1' / f'{COHORT}.json'
if not MANIFEST.is_file():
    raise FileNotFoundError(f'Frozen split manifest is missing: {MANIFEST}')
MANIFEST_SHA256 = hashlib.sha256(MANIFEST.read_bytes()).hexdigest()
TARGET_UNITS = {
    'Tg': 'K', 'density': 'unit not verified for Extended',
    'Tc': 'unit not verified', 'glass_cte': '10^-6 K^-1',
    'rubber_cte': '10^-6 K^-1',
}


## 2. Load and validate the frozen cohort

`load_cohort` performs the established target-specific cleaning directly from MongoDB. `load_manifest` verifies the saved split metadata and source fingerprint. `validate_frozen_splits` checks every saved outer and inner partition (outer fold IDs are 1–5).

On `Tg_md_common`, MD source CSVs are aligned by `polymer_id` to the manifest `sample_id` values. Only declared finite MD columns are used; no cohort filtering or new splitting occurs. The same validated cohort is reused for all selected representations, including optional MD-only ablations.

In the bulk-delta arm, the MD inputs are the reference values `end_to_end_md`, `Rg_md`, and `density_md` at 300 K, plus the 350 K minus 250 K difference of each property. The six values are derived without target data and replace the nine original bulk columns; no single-chain columns are included.

In [5]:
md_sources = (
    {name: MD_DIR / name for name in MD_COLUMNS}
    if require_md_coverage else None
)
with MongoClient(__import__('os').environ['MONGO_URI']) as mongo:
    frame, exclusions = load_cohort(
        dataset, target=target, md_dir=MD_DIR if require_md_coverage else None,
        client=mongo,
    )

manifest = load_manifest(MANIFEST, frame=frame, source_paths=md_sources)
ids, verified_folds = validate_frozen_splits(frame, manifest, fold_ids=fold_ids)
assert len(verified_folds) == manifest['n_outer_folds'] == 5
assert len(ids) == len(frame)

print(f'{COHORT}: {len(frame)} samples; {len(exclusions)} exclusions; manifest verified')
print(f'Descriptors: RDKit, Morgan, MACCS' + (' and Mordred' if USE_MORDRED else ''))
if require_md_coverage:
    md_hashes = md_source_hashes(MD_DIR)
    bulk = load_md_features(MD_DIR, ids, 'bulk')
    single = load_md_features(MD_DIR, ids, 'single_chain')
    bulk_delta = build_bulk_delta_features(bulk)
    assert bulk.shape == (len(ids), 9)
    assert single.shape == (len(ids), 8)
    assert bulk_delta.shape == (len(ids), 6)
    print(f'MD predictors: bulk={bulk.shape[1]}, single-chain={single.shape[1]}, bulk-delta={bulk_delta.shape[1]}')
else:
    md_hashes = None
print(f'Representations: {", ".join(REPRESENTATIONS)}')
print(f'Nested CV: 5 outer × 5 stored inner folds; full training enabled: {RUN_FULL_BENCHMARK}')
print(f'Target unit: {TARGET_UNITS[target]}')


Tg_md_common: 135 samples; 180 exclusions; manifest verified
Descriptors: RDKit, Morgan, MACCS
MD predictors: bulk=9, single-chain=8, bulk-delta=6
Representations: md_bulk, md_single_chain, md_bulk_delta
Nested CV: 5 outer × 5 stored inner folds; full training enabled: True
Target unit: K


## 3. Frozen nested cross-validation

The final model in each outer fold sees only that fold's training data. Within the inner search, the entire preprocessing and feature-selection pipeline is fitted again on each inner training subset. The held-out outer fold is used only for the final evaluation.

Outputs are OOF predictions indexed by `sample_id`, fold-by-fold metrics and hyperparameters, selected features and their stability, and elapsed training times. `MAE/σ` uses the standard deviation of the entire selected cohort as its denominator; this is an evaluation normalization, not a model input.

The structural branch calls the existing `benchmark_gbr()` with its original arguments. Every MD-only or structural+MD branch shares the same `FeatureFilter`, `MIPermutationSelector` and frozen nested CV splits through the appended `benchmark_gbr_md()` in the same `gbr.py` module. Each representation is logged as a separate run, so historical results are never overwritten.


In [6]:
if RUN_FULL_BENCHMARK:
    summaries = []
    for representation in REPRESENTATIONS:
        use_md = representation != 'structural'
        use_mordred = USE_MORDRED if not representation.startswith('md_') else False
        print(f'\nRunning {COHORT}: {representation}' + (' + Mordred' if use_mordred else ''))
        log = RunLogger(
            config={
                'db': DB_NAME, 'collection': COLLECTIONS[dataset], 'target': target,
                'cohort': COHORT, 'manifest': str(MANIFEST.relative_to(ROOT)),
                'manifest_sha256': MANIFEST_SHA256,
                'use_md': use_md, 'md_only': representation.startswith('md_'), 'use_mordred': use_mordred,
                **({'representation': representation, 'md_source_sha256': md_hashes} if use_md else {}),
                'target_unit': TARGET_UNITS[target],
                'random_state': SEED, 'param_grid': PARAM_GRID,
                'selection': 'training-only MI and OOF permutation importance',
            },
            tag=f'phase03_{COHORT}_{representation + "_" if use_md else ""}{datetime.now():%H%M%S_%f}',
        )
        log.dataset(db=DB_NAME, collection=COLLECTIONS[dataset], n_excluded=len(exclusions))
        with tempfile.TemporaryDirectory(prefix='gbr_cv_cache_') as cache:
            if not use_md:
                oof, folds, features, metrics = benchmark_gbr(
                    frame, manifest, fold_ids=fold_ids, use_mordred=use_mordred,
                    log=log, param_grid=PARAM_GRID, seed=SEED, cache_dir=cache,
                )
            else:
                oof, folds, features, metrics = benchmark_gbr_md(
                    frame, manifest, md_dir=MD_DIR, representation=representation,
                    fold_ids=fold_ids, use_mordred=use_mordred,
                    log=log, param_grid=PARAM_GRID, seed=SEED, cache_dir=cache,
                )
        assert hashlib.sha256(MANIFEST.read_bytes()).hexdigest() == MANIFEST_SHA256
        if use_md:
            assert md_source_hashes(MD_DIR) == md_hashes, 'MD source files changed during training'
        log.finish()
        summaries.append({'representation': representation, 'use_mordred': use_mordred, **metrics})
        display(pd.DataFrame([metrics]), folds)
        display(oof.head(), features.head())
    display(pd.DataFrame(summaries))
else:
    print('Training is disabled.')



Running Tg_md_common: md_bulk
[runlog] C:\Users\Vale\OneDrive\Desktop\uni\tirocinio\polymer_prediction\runs\11-22_1910_biceranoPolymers_md_mdonly_phase03_Tg_md_common_md_bulk_191041_029818
# Run `11-22_1910_biceranoPolymers_md_mdonly_phase03_Tg_md_common_md_bulk_191041_029818`

- **inizio**: 2026-11-22T19:10:41  |  **fine**: 2026-11-22T19:16:35  |  **durata**: 5.91 min
- **commit**: `b5e936b`
- **tag**: phase03_Tg_md_common_md_bulk_191041_029818

## Dati

- **sorgente**: MongoDB `PolymerPrediction.biceranoPolymers`
- **target**: Tg
- **polimeri usati**: 135
- **descrittori del monomero**: RDKit + Morgan + MACCS (`USE_MORDRED = False`)
- **range Tg**: 188 - 493 K (media 305, std 75)

## Feature MD

- **usate**: SI (0 feature)
- **sorgente**: `?`
- **lista**:

## Configurazione

- **cohort**: Tg_md_common
- **manifest**: data\splits\v1\Tg_md_common.json
- **manifest_sha256**: b5c05377824f071fdce4f8b0197084fde969cd3aaefedb12b260868620b81680
- **md_only**: True
- **use_mordred**: False
- 

C:\Users\Vale\OneDrive\Desktop\uni\tirocinio\polymer_prediction\runlog.py:377: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([pd.read_csv(index_path), df], ignore_index=True)


,n_samples,n_outer,n_inner,MAE,R2,MAE_over_sigma,fold_MAE_mean,fold_MAE_std,selection_jaccard,fit_seconds
0,135,5,5,44.515326,0.412649,0.594149,44.515326,7.241991,0.8,354.442814


,fold,n_train,n_test,n_selected,inner_MAE,MAE,R2,MAE_over_sigma,fit_seconds,best_params
0,1,108,27,2,42.671111,54.250359,0.099236,0.724083,72.271201,"{'gbr__learning_rate': 0.1, 'gbr__max_depth': ..."
1,2,108,27,2,51.421497,40.216402,0.547952,0.536771,72.382314,"{'gbr__learning_rate': 0.05, 'gbr__max_depth':..."
2,3,108,27,3,39.639400,45.522762,0.404731,0.607595,72.672947,"{'gbr__learning_rate': 0.1, 'gbr__max_depth': ..."
3,4,108,27,3,48.822333,35.179654,0.467900,0.469545,68.191205,"{'gbr__learning_rate': 0.1, 'gbr__max_depth': ..."
4,5,108,27,2,39.069826,47.407455,0.475733,0.632750,68.925147,"{'gbr__learning_rate': 0.05, 'gbr__max_depth':..."


,sample_id,fold,y_true,y_pred
0,698ddceeb7014ee539b3abeb,2,188.0,257.189702
1,698ddceeb7014ee539b3abec,5,190.0,258.199481
2,698ddceeb7014ee539b3abef,4,194.0,206.629533
3,698ddceeb7014ee539b3abf0,5,194.0,253.943304
4,698ddceeb7014ee539b3abf1,4,195.0,210.421771


,fold,feature
0,1,md__density_md_350
1,1,md__density_md_250
2,2,md__density_md_350
3,2,md__density_md_250
4,3,md__density_md_350


... (9 outputs left out)

,fold,n_train,n_test,n_selected,inner_MAE,MAE,R2,MAE_over_sigma,fit_seconds,best_params
0,1,108,27,1,30.193776,35.067011,0.588982,0.468042,68.296976,"{'gbr__learning_rate': 0.05, 'gbr__max_depth':..."
1,2,108,27,1,33.043045,29.430934,0.690136,0.392817,66.281523,"{'gbr__learning_rate': 0.01, 'gbr__max_depth':..."
2,3,108,27,1,32.143665,37.916335,0.518668,0.506072,66.842996,"{'gbr__learning_rate': 0.05, 'gbr__max_depth':..."
3,4,108,27,1,31.748680,32.391860,0.679185,0.432336,65.944388,"{'gbr__learning_rate': 0.01, 'gbr__max_depth':..."
4,5,108,27,1,30.785329,33.986937,0.615646,0.453626,66.216606,"{'gbr__learning_rate': 0.01, 'gbr__max_depth':..."


,sample_id,fold,y_true,y_pred
0,698ddceeb7014ee539b3abeb,2,188.0,215.869310
1,698ddceeb7014ee539b3abec,5,190.0,231.744225
2,698ddceeb7014ee539b3abef,4,194.0,209.940151
3,698ddceeb7014ee539b3abf0,5,194.0,228.131405
4,698ddceeb7014ee539b3abf1,4,195.0,240.563006


,fold,feature
0,1,md__density_md_delta_350_250
1,2,md__density_md_delta_350_250
2,3,md__density_md_delta_350_250
3,4,md__density_md_delta_350_250
4,5,md__density_md_delta_350_250


,representation,use_mordred,n_samples,n_outer,n_inner,MAE,R2,MAE_over_sigma,fold_MAE_mean,fold_MAE_std,selection_jaccard,fit_seconds
0,md_bulk,False,135,5,5,44.515326,0.412649,0.594149,44.515326,7.241991,0.8,354.442814
1,md_single_chain,False,135,5,5,48.346304,0.394721,0.645281,48.346304,3.339752,0.8,348.441481
2,md_bulk_delta,False,135,5,5,33.758615,0.622720,0.450578,33.758615,3.148314,1.0,333.582490
